# MRT-Labor: Station Auswertung

LU Biomedizinische Technik 2, Labor (717.402), TU Graz

In diesem Notebook werden MR-Daten im DICOM-Format eingelesen und ausgewertet:

* **A1** DICOM-Daten einlesen und darstellen (auch die eigenen Messdaten der Station Messung)
* **A2** Bestimmung von $T_1$ und $T_2$ eines Phantoms
* **A3** Einfluss des Rauschens auf die gefitteten Relaxationszeiten
* **A4** k-Raum, Auflösung und SNR

**Bedienung:** Die Zellen werden der Reihe nach mit `Shift + Enter` (oder dem Play-Symbol links) ausgeführt. Parameter, die verändert werden sollen, sind in Formularfeldern bzw. als Variablen in GROSSBUCHSTABEN am Anfang einer Zelle angegeben. Die für das Protokoll benötigten Abbildungen können per Rechtsklick gespeichert werden.

In [ ]:
#@title 0. Vorbereitung: Pakete und Hilfsfunktionen laden (einfach ausführen) { display-mode: "form" }
import sys, subprocess, os, glob, zipfile, io, urllib.request
try:
    import pydicom
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pydicom"], check=True)
    import pydicom
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from scipy.optimize import curve_fit

plt.rcParams.update({"figure.dpi": 100, "font.size": 10})
IN_COLAB = "google.colab" in sys.modules


# ---------- DICOM ----------------------------------------------------------
def _find_tag(ds, keyword):
    """Look up a DICOM attribute at top level and inside the functional groups
    (enhanced DICOM, e.g. newer Siemens scanners)."""
    if keyword in ds and ds.data_element(keyword).value not in (None, ""):
        return ds.data_element(keyword).value
    for group in ("SharedFunctionalGroupsSequence", "PerFrameFunctionalGroupsSequence"):
        if group in ds:
            stack = [ds.data_element(group).value[0]]
            while stack:
                item = stack.pop()
                for el in item:
                    if el.keyword == keyword and el.value not in (None, ""):
                        return el.value
                    if el.VR == "SQ" and len(el.value) > 0:
                        stack.append(el.value[0])
    return None


def _to_float(x):
    if x is None:
        return np.nan
    if isinstance(x, (list, tuple, pydicom.multival.MultiValue)):
        x = x[0] if len(x) else None
    try:
        return float(x)
    except (TypeError, ValueError):
        return np.nan


def load_dicom(path):
    """Read a DICOM file. Returns the image (float, rescale applied) and a dict
    with the most relevant header entries."""
    ds = pydicom.dcmread(path)
    img = ds.pixel_array.astype(float)
    slope = _to_float(_find_tag(ds, "RescaleSlope"))
    intercept = _to_float(_find_tag(ds, "RescaleIntercept"))
    if not np.isnan(slope):
        img = img * slope
    if not np.isnan(intercept):
        img = img + intercept
    te = _to_float(_find_tag(ds, "EchoTime"))
    if np.isnan(te):
        te = _to_float(_find_tag(ds, "EffectiveEchoTime"))
    ti = _to_float(_find_tag(ds, "InversionTime"))
    if np.isnan(ti):
        ti = _to_float(_find_tag(ds, "InversionTimes"))
    spacing = _find_tag(ds, "PixelSpacing")
    info = {
        "file": os.path.basename(path),
        "series": str(_find_tag(ds, "SeriesDescription") or ""),
        "sequence": str(_find_tag(ds, "SequenceName") or _find_tag(ds, "PulseSequenceName") or ""),
        "image_type": "\\".join(ds.ImageType) if "ImageType" in ds else "",
        "TE_ms": te,
        "TR_ms": _to_float(_find_tag(ds, "RepetitionTime")),
        "TI_ms": ti,
        "flip_angle_deg": _to_float(_find_tag(ds, "FlipAngle")),
        "matrix": "x".join(str(n) for n in img.shape),
        "pixel_size_mm": f"{_to_float(spacing[0]):.2f}" if spacing is not None else "",
        "slice_thickness_mm": _to_float(_find_tag(ds, "SliceThickness")),
        "scanner": f"{_find_tag(ds, 'Manufacturer') or ''} {_find_tag(ds, 'ManufacturerModelName') or ''}".strip(),
    }
    return img, info


def find_dicoms(folder):
    files = []
    for p in sorted(glob.glob(os.path.join(folder, "**", "*"), recursive=True)):
        if os.path.isfile(p):
            try:
                pydicom.dcmread(p, stop_before_pixels=True)
                files.append(p)
            except Exception:
                pass
    return files


def header_table(files):
    return pd.DataFrame([load_dicom(p)[1] for p in files])


def load_series(folder, sort_key):
    """Load all DICOMs in a folder, sorted by 'TE_ms' or 'TI_ms'."""
    data = [load_dicom(p) for p in find_dicoms(folder)]
    data.sort(key=lambda d: d[1][sort_key])
    images = np.stack([d[0] for d in data])
    values = np.array([d[1][sort_key] for d in data])
    return images, values, pd.DataFrame([d[1] for d in data])


def show_image(img, title="", ax=None, roi=None, **kw):
    if img.ndim == 3:                       # multi-slice / 3D data: show central slice
        img = img[img.shape[0] // 2]
    if ax is None:
        _, ax = plt.subplots(figsize=(4, 4))
    h = ax.imshow(img, cmap="gray", **kw)
    ax.set_title(title, fontsize=9)
    ax.axis("off")
    if roi is not None:
        y0, y1, x0, x1 = roi
        ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, ec="r", lw=1.5))
    return h


def show_series(images, labels, ncols=8, **kw):
    n = len(images)
    nrows = int(np.ceil(n / ncols))
    fig, axs = plt.subplots(nrows, ncols, figsize=(1.8 * ncols, 1.9 * nrows), squeeze=False)
    vmin = kw.pop("vmin", np.percentile(images, 1))
    vmax = kw.pop("vmax", np.percentile(images, 99.5))
    for i, ax in enumerate(axs.flat):
        if i < n:
            show_image(images[i], labels[i], ax=ax, vmin=vmin, vmax=vmax, **kw)
        else:
            ax.axis("off")
    plt.tight_layout()
    plt.show()


def auto_roi(img, fraction=0.4):
    """Square ROI in the centre of the object (threshold on |img|)."""
    a = np.abs(img)
    mask = a > 0.3 * np.percentile(a, 99)
    ys, xs = np.nonzero(mask)
    cy, cx = int(ys.mean()), int(xs.mean())
    h = int(fraction * (ys.max() - ys.min()) / 2)
    w = int(fraction * (xs.max() - xs.min()) / 2)
    return [cy - h, cy + h, cx - w, cx + w]


def roi_mean(images, roi):
    y0, y1, x0, x1 = roi
    return images[:, y0:y1, x0:x1].mean(axis=(1, 2))


# ---------- Signal models --------------------------------------------------
def model_t2(te, m0, t2):
    return m0 * np.exp(-te / t2)


def model_ir(ti, m0, t1):
    return m0 * (1 - 2 * np.exp(-ti / t1))


def model_ir_3p(ti, m0, t1, k):
    return m0 * (1 - k * np.exp(-ti / t1))


def fit(model, x, y, p0):
    p, cov = curve_fit(model, x, y, p0=p0, maxfev=20000)
    return p, np.sqrt(np.diag(cov))


# ---------- Data -----------------------------------------------------------
def unzip_bytes(zip_bytes, target):
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as z:
        z.extractall(target)


def upload_files(target):
    """File picker in Colab. ZIP archives are extracted."""
    from google.colab import files
    os.makedirs(target, exist_ok=True)
    for name, content in files.upload().items():
        if name.lower().endswith(".zip"):
            unzip_bytes(content, target)
        else:
            with open(os.path.join(target, name), "wb") as f:
                f.write(content)

print("Hilfsfunktionen geladen.")

In [ ]:
#@title Phantomdaten laden { display-mode: "form" }
#@markdown Die Phantomdaten werden automatisch heruntergeladen. Falls das nicht funktioniert, kann die ZIP-Datei aus dem TeachCenter nach dem Ausführen dieser Zelle manuell hochgeladen werden.
DATA_URL = "https://github.com/fglang/bmt-lab-mrt/raw/refs/heads/main/mrt_phantomdaten.zip"  #@param {type:"string"}
DATA_DIR = os.environ.get("MRI_DATA_DIR", "/content/phantom_data" if IN_COLAB else "phantom_data")

if not os.path.isdir(os.path.join(DATA_DIR, "IR")):
    try:
        with urllib.request.urlopen(DATA_URL, timeout=30) as r:
            unzip_bytes(r.read(), DATA_DIR)
    except Exception as e:
        print("Download nicht möglich:", e)
        if IN_COLAB:
            print("Bitte die Datei mri_phantom_data.zip auswählen:")
            upload_files(DATA_DIR)

print("Inhalt:", sorted(os.listdir(DATA_DIR)))

## A1: DICOM-Daten

MR-Bilder werden im Format DICOM (*Digital Imaging and Communications in Medicine*) gespeichert. Eine DICOM-Datei enthält neben den Bilddaten einen Header mit Informationen zu Patient bzw. Messobjekt, Scanner und Sequenzparametern. Neuere Scanner speichern häufig einen ganzen Datensatz (z. B. alle Schichten) in einer einzigen Datei (*Enhanced DICOM*).

Die folgende Zelle liest eine Datei der Multi-Echo-Spin-Echo-Messung ein und zeigt Bild und Header-Informationen.

In [ ]:
mse_files = find_dicoms(os.path.join(DATA_DIR, "MSE"))
img, info = load_dicom(mse_files[0])

h = show_image(img, info["series"])
plt.colorbar(h, fraction=0.046)
plt.show()
pd.Series(info)

Der vollständige Header kann mit `pydicom.dcmread(...)` angezeigt werden. Entfernen Sie dazu das `#` in der folgenden Zelle. Suchen Sie die Einträge für Echozeit, Repetitionszeit und Feldstärke.

In [ ]:
# pydicom.dcmread(mse_files[0])

### Eigene Messdaten

Die an der Station Messung aufgenommenen Bilder können hier hochgeladen werden (einzelne Dateien oder ein ZIP-Archiv mit Ordnern). Anschließend werden alle gefundenen DICOM-Dateien mit ihren Sequenzparametern aufgelistet.

In [ ]:
OWN_DATA_DIR = "/content/own_data" if IN_COLAB else "own_data"
if IN_COLAB:
    upload_files(OWN_DATA_DIR)
own_files = find_dicoms(OWN_DATA_DIR) if os.path.isdir(OWN_DATA_DIR) else []
own_table = header_table(own_files) if own_files else pd.DataFrame()
own_table

In der folgenden Zelle wird mit `INDEX` die Zeilennummer aus der Tabelle gewählt. Bei Mehrschicht-Datensätzen kann mit `SLICE` eine Schicht gewählt werden (`None` = mittlere Schicht).

In [ ]:
INDEX = 0
SLICE = None

if own_files:
    img, info = load_dicom(own_files[INDEX])
    if img.ndim == 3:
        img = img[img.shape[0] // 2 if SLICE is None else SLICE]
    show_image(img, f"{info['series']}: TE = {info['TE_ms']} ms, TR = {info['TR_ms']} ms")
    plt.show()

## A2: Bestimmung von $T_1$ und $T_2$

Gegeben sind zwei Messungen eines homogenen, flüssigkeitsgefüllten Phantoms:

* eine **Multi-Echo-Spin-Echo-Messung** (MSE) mit mehreren Echozeiten $T_E$ zur Bestimmung von $T_2$,
* eine **Inversion-Recovery-Serie** (IR) mit mehreren Inversionszeiten $T_I$ zur Bestimmung von $T_1$.

Für jede Serie wird die mittlere Signalintensität in einer Region of Interest (ROI) bestimmt und das jeweilige Signalmodell mit der Methode der kleinsten Quadrate angepasst (`scipy.optimize.curve_fit`).

### $T_2$ aus der Multi-Echo-Spin-Echo-Messung

In [ ]:
mse_images, te, mse_table = load_series(os.path.join(DATA_DIR, "MSE"), "TE_ms")
show_series(mse_images, [f"TE = {t:.0f} ms" for t in te])
mse_table[["file", "series", "TE_ms", "TR_ms"]]

Die ROI wird automatisch im Zentrum des Phantoms gewählt. Sie kann über `ROI = [y0, y1, x0, x1]` (Pixelkoordinaten) angepasst werden, z. B. um Artefakte am Rand auszuschließen.

In [ ]:
ROI = auto_roi(mse_images[0])     # or e.g. ROI = [80, 120, 70, 110]
print("ROI =", ROI)
show_image(mse_images[0], "ROI", roi=ROI)
plt.show()

mse_signal = roi_mean(mse_images, ROI)

Mit `EXCLUDE_FIRST_ECHO` wird festgelegt, ob der erste Echopunkt im Fit berücksichtigt wird.

In [ ]:
EXCLUDE_FIRST_ECHO = False   #@param {type:"boolean"}

first = 1 if EXCLUDE_FIRST_ECHO else 0
p, dp = fit(model_t2, te[first:], mse_signal[first:], p0=[mse_signal[0], 100])

t_fine = np.linspace(0, te.max(), 300)
fig, axs = plt.subplots(1, 2, figsize=(10, 3.8))
titles = {"linear": "lineare Darstellung", "log": "halblogarithmische Darstellung"}
for ax, scale in zip(axs, ["linear", "log"]):
    ax.plot(te, mse_signal, "o", label="Messdaten (ROI-Mittelwert)")
    if first:
        ax.plot(te[0], mse_signal[0], "x", ms=10, color="k", label="nicht im Fit")
    ax.plot(t_fine, model_t2(t_fine, *p), "--", label=f"Fit: $T_2$ = {p[1]:.0f} ± {dp[1]:.0f} ms")
    ax.set_xlabel("Echozeit $T_E$ / ms"); ax.set_ylabel("Signal / a.u.")
    ax.set_yscale(scale); ax.set_title(titles[scale])
axs[0].legend()
plt.tight_layout(); plt.show()
print(f"M0 = {p[0]:.1f} ± {dp[0]:.1f},  T2 = {p[1]:.1f} ± {dp[1]:.1f} ms")

**Aufgabe:** Führen Sie den Fit mit und ohne den ersten Messpunkt durch und vergleichen Sie die Ergebnisse. Die halblogarithmische Darstellung (rechts) ist hilfreich: Ein monoexponentieller Zerfall erscheint dort als Gerade.

### $T_1$ aus der Inversion-Recovery-Serie

Die IR-Bilder liegen als **Realteilbilder** vor (Bildtyp `R` im Header). Anders als in Betragsbildern bleibt das Vorzeichen der Magnetisierung erhalten, sodass direkt das Modell $S(T_I) = M_0\,(1 - 2\,e^{-T_I/T_1})$ angepasst werden kann. Mit `THREE_PARAMETER_FIT` wird statt des Faktors 2 ein freier Parameter $k$ gefittet, der eine unvollständige Inversion berücksichtigt.

In [ ]:
ir_images, ti, ir_table = load_series(os.path.join(DATA_DIR, "IR"), "TI_ms")
vmax = np.abs(ir_images).max()
show_series(ir_images, [f"TI = {t:.0f} ms" for t in ti], vmin=-vmax, vmax=vmax)
ir_table[["file", "series", "image_type", "TI_ms", "TR_ms", "TE_ms"]]

In [ ]:
ROI_IR = auto_roi(ir_images[0])
THREE_PARAMETER_FIT = False   #@param {type:"boolean"}

ir_signal = roi_mean(ir_images, ROI_IR)
if THREE_PARAMETER_FIT:
    model = model_ir_3p
    p, dp = fit(model, ti, ir_signal, p0=[ir_signal[-1], 500, 2])
else:
    model = model_ir
    p, dp = fit(model, ti, ir_signal, p0=[ir_signal[-1], 500])

t_fine = np.linspace(0, ti.max(), 300)
fig, axs = plt.subplots(1, 2, figsize=(10, 3.8), gridspec_kw={"width_ratios": [1, 1.6]})
show_image(ir_images[0], f"TI = {ti[0]:.0f} ms", ax=axs[0], roi=ROI_IR)
axs[1].plot(ti, ir_signal, "o", label="Messdaten (ROI-Mittelwert)")
axs[1].plot(t_fine, model(t_fine, *p), "--", label=f"Fit: $T_1$ = {p[1]:.0f} ± {dp[1]:.0f} ms")
axs[1].axhline(0, color="0.6", lw=0.8)
axs[1].set_xlabel("Inversionszeit $T_I$ / ms"); axs[1].set_ylabel("Signal / a.u.")
axs[1].legend()
plt.tight_layout(); plt.show()
print("Parameter (M0, T1 [, k]):", np.round(p, 2), "±", np.round(dp, 2))

## A3: Einfluss des Rauschens

Wie genau lässt sich $T_2$ aus verrauschten Daten bestimmen? Dazu wird ein Signalverlauf $S(T_E) = M_0\,e^{-T_E/T_2}$ simuliert und mit Rauschen überlagert. Die Simulation wird viele Male mit neuem Rauschen wiederholt, und jedes Mal wird $T_2$ gefittet. Mittelwert und Streuung der Fitergebnisse zeigen die systematische bzw. zufällige Abweichung vom wahren Wert.

Das Rauschen wird wie im Scanner auf Real- und Imaginärteil addiert. Mit `MAGNITUDE` wird anschließend der Betrag gebildet (Betragsbild), sonst wird der Realteil verwendet.

Parameter: wahres $T_2$ (`T2_TRUE`, in ms), SNR bezogen auf $M_0$ (`SNR`), Anzahl der Echos (`N_ECHOES`) und Echoabstand (`ECHO_SPACING`, in ms).

In [ ]:
T2_TRUE = 80          #@param {type:"number"}
SNR = 20              #@param {type:"slider", min:2, max:100, step:1}
N_ECHOES = 16         #@param {type:"slider", min:3, max:32, step:1}
ECHO_SPACING = 10     #@param {type:"number"}
MAGNITUDE = True      #@param {type:"boolean"}
N_REPETITIONS = 500

rng = np.random.default_rng()
te_sim = ECHO_SPACING * np.arange(1, N_ECHOES + 1)
m0 = 1.0
sigma = m0 / SNR
ideal = model_t2(te_sim, m0, T2_TRUE)

t2_fits = []
for _ in range(N_REPETITIONS):
    s = ideal + rng.normal(0, sigma, te_sim.size) + 1j * rng.normal(0, sigma, te_sim.size)
    s = np.abs(s) if MAGNITUDE else s.real
    try:
        p, _ = fit(model_t2, te_sim, s, p0=[s[0], T2_TRUE])
        t2_fits.append(p[1])
    except RuntimeError:
        pass
t2_fits = np.array(t2_fits)
t2_fits = t2_fits[(t2_fits > 0) & (t2_fits < 10 * T2_TRUE)]

fig, axs = plt.subplots(1, 2, figsize=(10, 3.6))
axs[0].plot(te_sim, ideal, "k-", label="ohne Rauschen")
axs[0].plot(te_sim, s, "o", label="ein verrauschter Datensatz")
axs[0].set_xlabel("$T_E$ / ms"); axs[0].set_ylabel("Signal / $M_0$"); axs[0].legend()
axs[1].hist(t2_fits, bins=40, color="0.6")
axs[1].axvline(T2_TRUE, color="k", ls="--", label="wahrer Wert")
axs[1].axvline(t2_fits.mean(), color="r", label="Mittelwert der Fits")
axs[1].set_xlabel("gefittetes $T_2$ / ms"); axs[1].set_ylabel("Anzahl"); axs[1].legend()
plt.tight_layout(); plt.show()
print(f"T2 = {t2_fits.mean():.1f} ± {t2_fits.std():.1f} ms "
      f"(wahr: {T2_TRUE} ms, Abweichung des Mittelwerts: {t2_fits.mean() - T2_TRUE:+.1f} ms)")

**Aufgaben:**
1. Verändern Sie das SNR. Wie ändern sich Streuung und Mittelwert der Fitergebnisse?
2. Verändern Sie Anzahl und Abstand der Echos bei festem $T_2$. Welche Echozeiten (bezogen auf $T_2$) liefern eine geringe Streuung? Was passiert, wenn alle Echos deutlich kürzer oder deutlich länger als $T_2$ sind?
3. Vergleichen Sie bei niedrigem SNR die Ergebnisse mit und ohne Betragsbildung. Woher kommt die systematische Abweichung?

## A4: k-Raum

Gemessen werden in der MRT nicht die Bildpunkte selbst, sondern die Fourier-Koeffizienten des Bildes (k-Raum). Hier wird der Zusammenhang in umgekehrter Richtung untersucht: Aus einem MR-Bild wird durch Fourier-Transformation der k-Raum berechnet, dieser wird verändert und anschließend wieder in ein Bild zurücktransformiert.

Als Testbild dient das erste Echo der MSE-Messung. Alternativ kann ein eigenes Bild verwendet werden (zweite Zeile der folgenden Zelle).

In [ ]:
test_img = mse_images[0]
# test_img = load_dicom(own_files[INDEX])[0]   # own image (for 3D data add [slice])

def to_kspace(img):
    return np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(img)))

def to_image(ksp):
    return np.fft.fftshift(np.fft.ifft2(np.fft.ifftshift(ksp)))

ksp = to_kspace(test_img)
fig, axs = plt.subplots(1, 2, figsize=(8, 4))
show_image(test_img, "Bild", ax=axs[0])
show_image(np.log10(np.abs(ksp) + 1e-3), "k-Raum (Betrag, logarithmisch)", ax=axs[1])
plt.show()

### Zentrum und Randbereiche des k-Raums

Mit `RADIUS` wird die Größe des k-Raum-Zentrums (in k-Raum-Punkten) festgelegt.

In [ ]:
RADIUS = 12   #@param {type:"slider", min:2, max:60, step:1}

ny, nx = ksp.shape
yy, xx = np.mgrid[-ny // 2:ny - ny // 2, -nx // 2:nx - nx // 2]
centre = np.sqrt(xx**2 + yy**2) <= RADIUS

fig, axs = plt.subplots(2, 3, figsize=(11, 7.5))
for col, (mask, title) in enumerate([(np.ones_like(centre), "vollständiger k-Raum"),
                                     (centre, f"nur Zentrum (r ≤ {RADIUS})"),
                                     (~centre, f"nur Randbereich (r > {RADIUS})")]):
    show_image(np.log10(np.abs(ksp * mask) + 1e-3), title, ax=axs[0, col])
    show_image(np.abs(to_image(ksp * mask)), "rekonstruiertes Bild", ax=axs[1, col])
plt.tight_layout(); plt.show()

### Auflösung, Rauschen und Mittelung

Eine geringere Auflösung entspricht einem kleineren gemessenen k-Raum (weniger k-Raum-Zeilen, also kürzere Messzeit). Zusätzlich wird Rauschen addiert und über mehrere Wiederholungen gemittelt. Das SNR wird in einer ROI im Phantom bestimmt.

Parameter: Matrixgröße (`MATRIX`), relatives Rauschniveau (`NOISE_LEVEL`) und Anzahl der Mittelungen (`N_AVERAGES`).

In [ ]:
MATRIX = 64          #@param {type:"slider", min:16, max:192, step:8}
NOISE_LEVEL = 0.05   #@param {type:"number"}
N_AVERAGES = 1       #@param {type:"slider", min:1, max:16, step:1}

rng = np.random.default_rng(1)
signal_max = np.percentile(np.abs(test_img), 99)

def simulate_acquisition(matrix, n_avg):
    """Simulate an acquisition with the given matrix size (number of k-space lines
    and columns) and number of averages. Every k-space sample gets the same noise."""
    cy, cx = ny // 2, nx // 2
    h = matrix // 2
    ksp_crop = ksp[cy - h:cy + h, cx - h:cx + h]
    sigma_k = NOISE_LEVEL * signal_max * np.sqrt(ny * nx)
    acc = 0
    for _ in range(n_avg):
        noise = sigma_k * (rng.normal(size=ksp_crop.shape) + 1j * rng.normal(size=ksp_crop.shape)) / np.sqrt(2)
        acc = acc + ksp_crop + noise
    return np.abs(to_image(acc / n_avg)) * (matrix / ny) * (matrix / nx)   # scale to signal per area

def estimate_snr(img):
    y0, y1, x0, x1 = auto_roi(img, fraction=0.4)
    background = img[:max(3, img.shape[0] // 10), :max(3, img.shape[1] // 10)]
    return img[y0:y1, x0:x1].mean() / (background.std() / 0.655)   # Rayleigh correction

full = simulate_acquisition(min(ny, nx) // 2 * 2, 1)
reduced = simulate_acquisition(MATRIX, N_AVERAGES)
rel_time = MATRIX * N_AVERAGES / full.shape[0]
fig, axs = plt.subplots(1, 2, figsize=(9, 4.5))
show_image(full, f"Matrix {full.shape[0]}, 1 Mittelung\nSNR ≈ {estimate_snr(full):.0f}, rel. Messzeit 1", ax=axs[0])
show_image(reduced, f"Matrix {MATRIX}, {N_AVERAGES} Mittelung(en)\nSNR ≈ {estimate_snr(reduced):.0f}, rel. Messzeit {rel_time:.2f}", ax=axs[1])
plt.tight_layout(); plt.show()

**Aufgaben:**
1. Welche Bildeigenschaften werden durch das k-Raum-Zentrum, welche durch die Randbereiche bestimmt?
2. Verringern Sie die Matrixgröße. Was passiert mit Bildschärfe und SNR? Welche Strukturen erscheinen an den Kanten des Phantoms?
3. Stellen Sie eine Kombination aus Matrixgröße und Mittelungen ein, die bei kleinerer Matrix dasselbe SNR wie das linke Bild erreicht oder bei voller Matrix das SNR verdoppelt. Wie ändert sich jeweils die relative Messzeit (Messzeit proportional zu Anzahl der k-Raum-Zeilen mal Mittelungen)?